# Meaning head
A small network that reads Qwen's hidden state just before the last word of a line and predicts the GloVe
vector of the word it "means". The line end is then picked from the 300 vocabulary words closest to that vector
(in `generate.ipynb`).

Steps:
1. vocabulary: words in both GloVe and CMUdict
2. training examples: hidden state before each line's last word + that word
3. train the head and check it

Output: `lora_weights/vocab.npz` (words + their GloVe vectors), `lora_weights/meaning_head.pt`

In [1]:
%%capture
%run rhyme.ipynb

## 1. Vocabulary
GloVe (Wikipedia + news, 400k words) is sorted by how common a word is. We go down the list and keep words
that are only letters and have a pronunciation (CMUdict + our patch), until we have 50k.

News-common isn't poetry-common though: the 50k cut lets in *eurobonds* but not *thine* or *morn*. So every word
that ends a line at least 5 times in the training poems is added too (if GloVe has it), and the old words
`o'er`, `e'er` and `ne'er` (they have GloVe vectors).

In [2]:
import gzip
import numpy as np

N_WORDS = 50000
# line ends in the training poems
train_ends = Counter()
for text in train["text"]:
    for line in poem_lines(text):
        w = words_of(line)
        if w:
            train_ends[w[-1]] += 1
EXTRA = {w for w, n in train_ends.items() if n >= 5 and phones(w)} | {"o'er", "e'er", "ne'er"}

words, vectors, extra = [], [], {}
with gzip.open(DATA + "glove/glove-wiki-gigaword-300.gz", "rt", encoding="utf-8") as f:
    f.readline()  # header: 400000 300
    for line in f:
        word, rest = line.split(" ", 1)
        if len(words) < N_WORDS and re.fullmatch(r"[a-z]+", word) and phones(word):
            words.append(word)
            vectors.append(np.array(rest.split(), dtype=np.float32))
        elif word in EXTRA:
            extra[word] = np.array(rest.split(), dtype=np.float32)

extra = {w: v for w, v in extra.items() if w not in set(words)}
print(len(extra), "poetry words added, e.g.", [w for w in ["thine", "morn", "o'er", "e'er", "ne'er", "thee"] if w in extra])
for w, v in extra.items():
    words.append(w)
    vectors.append(v)

vectors = np.stack(vectors)
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)  # unit length, so a dot product is the cosine
print(len(words), "words")

2544 poetry words added, e.g. ['thine', 'morn', "o'er", "e'er", "ne'er"]
52544 words


How much of what poets actually end lines with is in there? (line ends in the val poems, so not the ones we added from)

In [3]:
in_vocab = set(words)
covered = sum(n for w, n in ends.items() if w in in_vocab)
print(f"{covered / sum(ends.values()):.1%} of line ends are in the vocabulary")
print("most common ones that aren't:", [w for w, n in ends.most_common(300) if w not in in_vocab][:25])

92.8% of line ends are in the vocabulary
most common ones that aren't: []


Nearest words to a few poetry words. GloVe learned from news and Wikipedia, so the neighbours are not
always poetic (something to mention in the report).

In [4]:
index = {w: i for i, w in enumerate(words)}


def nearest(word, n=8):
    sims = vectors @ vectors[index[word]]
    return [words[i] for i in np.argsort(-sims)[1:n + 1]]


for w in ["heart", "sea", "night", "rose", "death", "o'er"]:
    if w in index:
        print(f"{w:>6}: {', '.join(nearest(w))}")

 heart: brain, cardiac, blood, surgery, liver, cancer, coronary, kidney
   sea: ocean, waters, seas, mediterranean, coast, coastal, aegean, arctic
 night: evening, saturday, sunday, afternoon, morning, nights, weekend, friday
  rose: fell, climbed, surged, jumped, soared, percent, slipped, gained
 death: died, deaths, murder, dead, killed, execution, victim, killing
  o'er: faithfulness, greenness, forsaken, crucify, blest, exclaiming, loveliness, amazes


Rhyme richness over this vocabulary (how many words rhyme perfectly with each word), used when a line
opens a new rhyme.

In [5]:
rich_vocab = richness(words)
for w in ["light", "sea", "love", "heart", "silver", "orange", "month"]:
    print(w, rich_vocab.get(w))

np.savez("../lora_weights/vocab.npz", words=np.array(words), vectors=vectors.astype(np.float16),
         richness=np.array([rich_vocab[w] for w in words]))
print("saved", len(words), "words")

light 68
sea 272
love 8
heart 20
silver 1
orange 0
month 0
saved 52544 words


## 2. Training examples
Run the fine-tuned model over training poems and keep, for every line, the hidden state just before the line's
last word (the state Qwen uses to predict that word) and the word itself.

Only lines whose last word could be picked by the generator count: in the vocabulary, has a vowel, and ends at
least 5 training lines (the same rule as in `generate.ipynb`). The LoRA weights are merged into the model, the
generator does the same so the hidden states match.

In [6]:
import bisect
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

N_TRAIN_PIECES = 11000  # ~300k examples
N_VAL_PIECES = 1000
BATCH = 8

allowed = [w for w in words if re.search("[aeiouy]", w) and train_ends[w] >= 5]
allowed_idx = {w: i for i, w in enumerate(allowed)}
print(len(allowed), "words can end a line")

torch.set_grad_enabled(False)
tokenizer = AutoTokenizer.from_pretrained("../model/Qwen3-0.6B-Base")
model = AutoModelForCausalLM.from_pretrained("../model/Qwen3-0.6B-Base", dtype=torch.bfloat16).to("cuda")
model = PeftModel.from_pretrained(model, "../lora_weights/").merge_and_unload().eval()


def line_end_positions(text):
    # (character where the line's last word starts, that word) for every line after the header
    start = text.index("\n\n") + 2
    for m in re.finditer(r"[^\n]+", text[start:]):
        found = list(re.finditer(r"[A-Za-z][A-Za-z'’]*", m.group()))
        w = words_of(m.group())
        if found and w and words_of(found[-1].group())[-1:] == w[-1:]:
            yield start + m.start() + found[-1].start(), w[-1]


def extract(texts):
    hidden, target = [], []
    for b in range(0, len(texts), BATCH):
        batch = texts[b:b + BATCH]
        encs = [tokenizer(t, return_offsets_mapping=True) for t in batch]
        longest = max(len(e["input_ids"]) for e in encs)
        ids = torch.full((len(batch), longest), tokenizer.pad_token_id)
        mask = torch.zeros((len(batch), longest), dtype=torch.long)
        for i, e in enumerate(encs):
            ids[i, :len(e["input_ids"])] = torch.tensor(e["input_ids"])
            mask[i, :len(e["input_ids"])] = 1
        h = model.model(input_ids=ids.cuda(), attention_mask=mask.cuda()).last_hidden_state
        for i, (text, e) in enumerate(zip(batch, encs)):
            starts = [s for s, _ in e["offset_mapping"]]
            for pos, w in line_end_positions(text):
                t = bisect.bisect_right(starts, pos) - 1  # the token the word starts in
                if w in allowed_idx and t >= 1:
                    hidden.append(h[i, t - 1].half().cpu())
                    target.append(allowed_idx[w])
    return torch.stack(hidden), torch.tensor(target)


STATES = DATA + "line_end_states.pt"  # ~650 MB, saved so this only runs once (~15 min)
if os.path.exists(STATES):
    train_h, train_y, val_h, val_y = torch.load(STATES)
else:
    val_text = pd.read_parquet(DATA + "train_test/val.parquet", columns=["text"])
    train_h, train_y = extract(list(train["text"].sample(N_TRAIN_PIECES, random_state=0)))
    val_h, val_y = extract(list(val_text["text"].sample(N_VAL_PIECES, random_state=0)))
    torch.save((train_h, train_y, val_h, val_y), STATES)
print(len(train_y), "training examples,", len(val_y), "val examples")
print("most common targets:", Counter(allowed[i] for i in train_y.tolist()).most_common(10))

W1120 19:55:08.391000 11920 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


18395 words can end a line


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 465.24it/s]


297165 training examples, 27733 val examples
most common targets: [('me', 2740), ('day', 2172), ('away', 1480), ('be', 1448), ('night', 1398), ('again', 1285), ('eyes', 1247), ('there', 1125), ('light', 1124), ('thee', 1121)]


## 3. Train the head
1024 -> 1024 -> 300, its output compared with the GloVe vectors of all the line-end words (GloVe itself doesn't
change). Loss: InfoNCE, i.e. softmax over cosine / temperature, so the right word has to be closer than all the
others.

In [7]:
from torch import nn

TAU = 0.05
EPOCHS = 4
torch.set_grad_enabled(True)
torch.manual_seed(0)

V = torch.tensor(vectors[[index[w] for w in allowed]], dtype=torch.float32, device="cuda")
V = V / V.norm(dim=1, keepdim=True)

head = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, 1024), nn.GELU(), nn.Linear(1024, 300)).cuda()
opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=0.01)


def scores(h):
    v = head(h.cuda().float())
    return (v / v.norm(dim=1, keepdim=True)) @ V.T  # cosine with every line-end word


for epoch in range(EPOCHS):
    order = torch.randperm(len(train_y))
    total = 0
    for b in range(0, len(order), 1024):
        i = order[b:b + 1024]
        loss = nn.functional.cross_entropy(scores(train_h[i]) / TAU, train_y[i].cuda())
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(i)
    val_loss = 0
    with torch.no_grad():
        for b in range(0, len(val_y), 4096):
            loss = nn.functional.cross_entropy(scores(val_h[b:b + 4096]) / TAU, val_y[b:b + 4096].cuda(), reduction="sum")
            val_loss += loss.item()
    print(f"epoch {epoch + 1}: train loss {total / len(train_y):.3f}, val loss {val_loss / len(val_y):.3f}")
torch.set_grad_enabled(False)

epoch 1: train loss 6.653, val loss 6.173
epoch 2: train loss 5.896, val loss 5.970
epoch 3: train loss 5.649, val loss 5.899
epoch 4: train loss 5.480, val loss 5.853


torch.autograd.grad_mode.set_grad_enabled(mode=False)

## Is it any good?
How often is the real last word among the head's nearest words? Compared with Qwen's own most likely next
words (single-token line-end words only, like B1 uses) and with just guessing the most common line ends.

In [8]:
# Qwen tokens that are a whole line-end word (" light", " Light"), and which word
vocab_tokens = [tokenizer.decode([i]) for i in range(len(tokenizer))]
tok_ids, tok_word = [], []
for i, s in enumerate(vocab_tokens):
    if re.fullmatch(r" [A-Za-z]+", s) and s[1:].lower() in allowed_idx:
        tok_ids.append(i)
        tok_word.append(allowed_idx[s[1:].lower()])
tok_ids = torch.tensor(tok_ids, device="cuda")
tok_word = torch.tensor(tok_word, device="cuda")
print(len(tok_ids), "tokens are line-end words, the other words can only come from the head")

common = torch.bincount(train_y, minlength=len(allowed)).float().cuda()  # how often each word ends a line

hits = {"meaning head": [0, 0, 0], "Qwen's words": [0, 0, 0], "most common": [0, 0, 0]}
for b in range(0, len(val_y), 1024):
    h = val_h[b:b + 1024]
    y = val_y[b:b + 1024].cuda()

    p = torch.softmax(model.lm_head(h.cuda().to(torch.bfloat16)).float(), dim=1)[:, tok_ids]
    qwen = torch.zeros(len(h), len(allowed), device="cuda")
    qwen.index_add_(1, tok_word, p)  # " light" + " Light" both count for "light"

    for name, s in [("meaning head", scores(h)), ("Qwen's words", qwen), ("most common", common.repeat(len(h), 1))]:
        top = s.topk(300, dim=1).indices
        for j, k in enumerate([1, 10, 300]):
            hits[name][j] += (top[:, :k] == y[:, None]).any(dim=1).sum().item()

for name, n in hits.items():
    print(f"{name:<13} top 1: {n[0] / len(val_y):.1%}, top 10: {n[1] / len(val_y):.1%}, top 300: {n[2] / len(val_y):.1%}")

16708 tokens are line-end words, the other words can only come from the head
meaning head  top 1: 19.0%, top 10: 39.0%, top 300: 71.4%
Qwen's words  top 1: 22.5%, top 10: 48.6%, top 300: 81.5%
most common   top 1: 0.8%, top 10: 4.7%, top 300: 43.8%


In [9]:
for i in range(8):
    top = scores(val_h[i:i + 1])[0].topk(5).indices.tolist()
    print(f"real: {allowed[val_y[i]]:<10} head: {', '.join(allowed[j] for j in top)}")

torch.save({"head": head.state_dict(), "words": allowed, "tau": TAU}, "../lora_weights/meaning_head.pt")
print("saved")

real: traffic    head: began, children, started, workers, around
real: carriage   head: wheel, carriage, wagon, horse, truck
real: bargain    head: wages, wage, labor, toil, goods
real: losses     head: woes, troubles, worries, struggles, misery
real: rivers     head: rivers, river, sea, waters, stream
real: rivers     head: rivers, streams, pools, lakes, waterfalls
real: people     head: children, boys, young, girls, poor
real: city       head: towns, cities, highways, city, town
saved
